# visualizeLOB — 订单簿动态可视化 Demo

本 notebook 演示如何使用 `visualize_lob.py` 模块：
1. 生成符合逻辑的 toy 数据
2. 加载与筛选数据
3. 单帧静态展示（含差异高亮）
4. 逐帧动态回放动画

## 1. 导入依赖与设置

In [1]:
# 导入标准库
import os                                                          # 文件路径操作
from pathlib import Path                                           # 路径处理

# 导入数据处理库
import pandas as pd                                                # 数据分析
import numpy as np                                                 # 数值计算

# 导入可视化库
import plotly.graph_objects as go                                  # Plotly 图形对象

# 导入自定义 LOB 可视化模块（所有 class 和函数都在这个文件里）
from visualize_lob import (                                        # 从 visualize_lob.py 导入
    generate_toy_data,                                             # toy 数据生成函数
    LOBDataLoader,                                                 # 数据加载器
    LOBVisualizer,                                                 # 可视化器
)

# 打印版本信息确认环境正常
print(f"pandas:  {pd.__version__}")                                # pandas 版本
print(f"numpy:   {np.__version__}")                                # numpy 版本
print(f"plotly:  {go.__version__}" if hasattr(go, '__version__') else "plotly: OK")  # plotly 版本

pandas:  3.0.2
numpy:   2.4.4
plotly: OK


## 2. 生成 Toy 数据（orderbook.parquet & triggerInfo.parquet）

调用 `generate_toy_data()` 函数，在 `toy_data/` 文件夹下生成符合真实市场逻辑的模拟数据：
- 初始化一个 10 档深度的订单簿（买一 ≈ 10.00，卖一 ≈ 10.01）
- 随机产生 100 个事件（被动挂单、主动吃单、撤单），逐帧记录状态变化
- 所有相邻帧的变化均符合限价订单簿规则

In [2]:
# ---- 生成 toy 数据 ----
toy_dir = "./toy_data"                                             # toy 数据输出目录

ob_path, trigger_path = generate_toy_data(                         # 调用生成函数
    output_dir=toy_dir,                                            # 输出目录
    code="000001",                                                 # 股票代码
    n_events=100,                                                  # 目标帧数
    seed=42,                                                       # 随机种子（可复现）
)

print(f"orderbook 文件:   {ob_path}")                              # 确认路径
print(f"triggerInfo 文件: {trigger_path}")                          # 确认路径

已生成 101 帧订单簿数据 → ./toy_data\orderbook.parquet
已生成 100 条触发信息 → ./toy_data\triggerInfo.parquet
orderbook 文件:   ./toy_data\orderbook.parquet
triggerInfo 文件: ./toy_data\triggerInfo.parquet


In [3]:
ob = pd.read_parquet('./toy_data/orderbook.parquet')
ti=pd.read_parquet('./toy_data/triggerInfo.parquet')

In [11]:
ob[['bidPx1', 'bidVlm1', 'askVlm1', 'askPx1', 'adjIndex']]

,bidPx1,bidVlm1,askVlm1,askPx1,adjIndex
0,10.00,580,420,10.02,0
1,10.00,580,420,10.02,1
2,10.00,580,420,10.02,2
3,10.00,580,130,10.02,3
4,10.02,10,300,10.03,4
...,...,...,...,...,...
96,10.03,70,10,10.07,96
97,10.03,70,10,10.07,97
98,10.03,70,10,10.07,98
99,10.03,20,10,10.07,99


In [12]:
ti

,code,adjIndex,triggerType
0,000001,1,cancel
1,000001,2,order
2,000001,3,order
3,000001,4,order
4,000001,5,order
...,...,...,...
95,000001,96,order
96,000001,97,order
97,000001,98,order
98,000001,99,order


## 3. 加载与查看数据

使用 `LOBDataLoader` 加载 parquet 文件，按股票代码筛选，查看数据概况。

In [4]:
# ---- 创建数据加载器 ----
loader = LOBDataLoader(                                            # 实例化加载器
    orderbook_path=ob_path,                                        # 订单簿文件路径
    trigger_path=trigger_path,                                     # 触发信息文件路径
)

# ---- 按股票代码筛选 ----
loader.filter(code="000001")                                       # 只看 000001

# ---- 打印基本信息 ----
print(f"总帧数: {loader.n_frames}")                                 # 帧数
print(f"adjIndex 范围: {loader.indices[0]} ~ {loader.indices[-1]}") # 索引范围
print()

# ---- 查看前 3 帧数据 ----
print("=== 前 3 帧订单簿数据 ===")                                  # 标题
for i in range(min(3, loader.n_frames)):                           # 遍历前 3 帧
    frame = loader.get_frame(i)                                    # 取帧
    trigger = loader.get_trigger(i)                                # 取触发类型
    print(f"\n帧 #{int(frame['adjIndex'])} | 触发: {trigger or '初始状态'}")  # 帧信息
    print(f"  买一: {frame['bidPx1']:.2f} × {int(frame['bidVlm1'])}")         # 买一
    print(f"  卖一: {frame['askPx1']:.2f} × {int(frame['askVlm1'])}")         # 卖一

总帧数: 101
adjIndex 范围: 0 ~ 100

=== 前 3 帧订单簿数据 ===

帧 #0 | 触发: 初始状态
  买一: 10.00 × 580
  卖一: 10.02 × 420

帧 #1 | 触发: cancel
  买一: 10.00 × 580
  卖一: 10.02 × 420

帧 #2 | 触发: order
  买一: 10.00 × 580
  卖一: 10.02 × 420


## 4. 单帧静态可视化（堆叠式 base + delta，含分割线）

展示第 10 帧订单簿的快照，每个价位的柱子分为两段堆叠：
- **底部 base**（标准色）：与上一帧相同的不变量
- **顶部 delta**（深色/浅色 + 分割线）：本帧发生的变化量
  - **深色** = 新增量（挂单 / partial fill 后余量）
  - **浅色半透明** = 减少量 ghost（撤单 / 被成交）
- **分割线**：base 与 delta 之间的深灰色边框，清晰区分
- **右上角标签**醒目显示该帧的 **triggerType**（ORDER / CANCEL）

In [5]:
# ---- 创建可视化器 ----
viz = LOBVisualizer(loader)                                        # 传入加载器

# ---- 绘制第 10 帧的单帧静态图 ----
fig_single = viz.plot_single_frame(pos=10)                         # 位置 10（第 11 帧）
fig_single.show()                                                  # 显示交互式图表

## 5. 逐帧动态回放动画（堆叠 + 分割线 + triggerType）

动画特性：
- **同价格位置固定不动**（全局统一 categoryarray），变化在原地展示
- 每帧用 **base + delta 堆叠** 展示变化，delta 段有分割线
- **右上角标签**随帧切换实时更新 triggerType（ORDER / CANCEL）
- 点击 **▶ 播放** 自动回放，**⏸ 暂停** 停在当前帧
- 拖动底部 **滑块** 跳转到任意帧
- 支持缩放/平移查看感兴趣的价格区间

In [6]:
# ---- 创建前 50 帧的动态回放动画 ----
fig_anim = viz.plot_animation(                                     # 调用动画方法
    start_pos=0,                                                   # 从第 0 帧开始
    end_pos=50,                                                    # 到第 50 帧
    frame_duration=600,                                            # 每帧 600 毫秒
)
fig_anim.show()                                                    # 显示动画

In [12]:
ob.loc[3:5, ['adjIndex', 'bidPx3', 'bidPx2', 'bidPx1', 'bidVlm1', 'askVlm1', 'askPx1', 'askPx2', 'askPx3', ]]

,adjIndex,bidPx3,bidPx2,bidPx1,bidVlm1,askVlm1,askPx1,askPx2,askPx3
3,3,9.98,9.99,10.00,580,130,10.02,10.03,10.04
4,4,9.99,10.00,10.02,10,300,10.03,10.04,10.06
5,5,9.99,10.00,10.02,10,300,10.03,10.04,10.05


## 6. 按索引范围筛选并可视化

演示数据筛选功能：只看 adjIndex 20~40 的帧，重新生成动画。

In [ ]:
# ---- 重新加载并筛选 adjIndex 20~40 ----
loader2 = LOBDataLoader(ob_path, trigger_path)                     # 新建加载器
loader2.filter(code="000001", start_index=20, end_index=40)        # 筛选范围

print(f"筛选后帧数: {loader2.n_frames}")                            # 确认帧数
print(f"adjIndex 范围: {loader2.indices[0]} ~ {loader2.indices[-1]}")  # 确认范围

# ---- 可视化筛选后的全部帧 ----
viz2 = LOBVisualizer(loader2)                                      # 新建可视化器
fig_filtered = viz2.plot_animation(frame_duration=500)             # 创建动画
fig_filtered.show()                                                # 显示